In [1]:
import os
import subprocess

# Find the root of the current git repository (usually your uv project root)
root = 'f:\\Projects\\GeneIndex'
os.chdir(root)

print(f"Current working directory: {os.getcwd()}")

Current working directory: f:\Projects\GeneIndex


In [2]:
from src.u1_downloader.images_downloader import download_group, get_all_gids
from src.u1_downloader.images_loader import load_all
import pandas as pd
from pathlib import Path
import torch
import torchvision
import matplotlib.pyplot as plt
import torchvision.transforms.functional as F
from PIL import Image
from tqdm import tqdm
from torchvision.transforms import v2
import h5py
import numpy as np
import shutil

In [3]:
TEMP_PATH = Path('src/m0_ae/dataset/temp')
OUT_PATH = Path('src/m0_ae/dataset/train_100k')

In [4]:
gids = get_all_gids()

In [5]:
pd.Series(gids).sample(5)

2193    1258
548     4079
1154    4862
3769    2918
919     2409
dtype: int64

In [6]:
def create_hdf5_dataset(input_folder, h5_path, patch_size=512, stride=400, batch_size=100):
    all_files = [os.path.join(input_folder, f) for f in os.listdir(input_folder) if f.endswith(('.jpg', '.png'))]
    
    with h5py.File(h5_path, 'w') as f:
        dst = f.create_dataset(
            'patches', 
            shape=(0, 3, patch_size, patch_size), 
            maxshape=(None, 3, patch_size, patch_size), 
            dtype=np.uint8,
            chunks=(1, 3, patch_size, patch_size)
        )
        
        temp_batch = []
        total_saved = 0
        
        for idx, img_path in tqdm(enumerate(all_files), total=len(all_files), desc='Patching images'):
            img = Image.open(img_path).convert("RGB")
            img = img.resize((img.width // 2, img.height // 2))
            
            for y in range(0, img.height - patch_size, stride):
                for x in range(0, img.width - patch_size, stride):
                    box = (x, y, x + patch_size, y + patch_size)
                    patch = img.crop(box)
                    
                    patch_np = np.array(patch).transpose(2, 0, 1)
                    temp_batch.append(patch_np)
                    
                    if len(temp_batch) >= batch_size:
                        arr_batch = np.array(temp_batch)
                        
                        dst.resize(total_saved + len(arr_batch), axis=0)
                        dst[total_saved:] = arr_batch
                        
                        total_saved += len(arr_batch)
                        temp_batch = []
                
        if len(temp_batch) > 0:
            arr_batch = np.array(temp_batch)
            dst.resize(total_saved + len(arr_batch), axis=0)
            dst[total_saved:] = arr_batch
            total_saved += len(arr_batch)

    print(f"Created file {h5_path} with {total_saved} patches.")

In [15]:
def do_shard(gid): 
    if not TEMP_PATH.exists():
        os.mkdir(TEMP_PATH)

    download_group(str(TEMP_PATH), gid)
    create_hdf5_dataset(TEMP_PATH, OUT_PATH / f'gid{gid}.h5')

    shutil.rmtree(TEMP_PATH)

In [17]:
do_shard(4862)

Found 24 subgroups in 4862


Patching images: 100%|██████████| 1155/1155 [03:51<00:00,  4.98it/s]


Created file src\m0_ae\dataset\train_100k\gid4862.h5 with 13588 patches.
